In [ ]:
from collections import defaultdict
from typing import List, Dict, Tuple
# 재현성
import random
random.seed(42)

In [9]:
# 작은 말뭉치
corpus = [
    "hello world hello there",
    "hello beautiful world",
    "the quick brown fox",
    "the lazy dog",
    "hello hello",
    "world peace"
]

class BPETokenizer:
    '''
    BPE(Byte-Pair-Encoding)

    작동:
    1. 텍스트를 문자 단위로 분해
    2. 가장 자주 나타나는 인접한 쌍(pair)을 찾음
    3. 그 쌍을 하나의 새 토큰으로 병합
    4. 2-3을 반복해서 어휘크기를 조절
    '''
    def __init__(self):
        self.word_freqs = defaultdict(int) # 단어빈도
        self.merges = {}  # 병합규칙(pair) -> new token
        self.vocab = set() # 현재 어휘
        self.merge_history = []  # 병합과정 기록
    def _build_initial_vocab(self, corpus:List[str]):
        '''step 1 : 텍스트를 단어어로 분해하고 문자 단위로 분리'''
        for text in corpus:
            words = text.split()
            for word in words:
                self.word_freqs[word] += 1
        
        # 초기어휘 : 모든 문자
        for word in self.word_freqs:
            for char in word:
                self.vocab.add(char)
        self.vocab.add('</w>')  # 특수토큰 : 단어 끝을 표시(GPT 스타일)
        return self
    
    def get_stats(self, vocab:Dict[str,int]) -> Dict[Tuple[str,str], int]:
        '''각(pair) 조합의 빈도를 계산'''
        pairs = defaultdict(int)
        for word, freq in vocab.items():
            symbols = word.split()
            for i in range(len(symbols)-1):
                pairs[symbols[i], symbols[i+1]] += freq              
        return pairs
    
    def merge_vocab(self, pair:Tuple[str,str], vocab:Dict[str,int]) -> Dict[str,int]:
        '''특정 pair를 하나의 토큰으로 병합'''
        new_vocab = {}
        bigram =' '.join(pair)
        replacement = ''.join(pair)
        for word in vocab:
            new_word = word.replace(bigram, replacement)
            new_vocab[new_word] = vocab[word]
        return new_vocab
    
    def train(self, corpus: List[str], num_merges: int = 10):
        """BPE 훈련: num_merges 번만큼 반복"""
        self._build_initial_vocab(corpus)
        
        # 초기 vocab dict 생성
        vocab = {}
        for word, freq in self.word_freqs.items():
            #  각 문자 사이에 공백 추가
            vocab[' '.join(word) + ' </w>'] = freq
        
        print(f"[초기 어휘 크기] {len(self.vocab)}")
        print(f"[병합 시작] {num_merges}번 반복\n")
        
        for i in range(num_merges):
            pairs = self.get_stats(vocab)
            
            if not pairs:
                print(f"  반복 {i+1}: 병합 가능한 pair가 없음. 중단.")
                break
            
            best_pair = max(pairs, key=pairs.get)
            best_freq = pairs[best_pair]
            
            # 각 병합 단계 상세 기록
            vocab = self.merge_vocab(best_pair, vocab)
            self.merges[best_pair] = ''.join(best_pair)
            new_token = ''.join(best_pair)
            self.vocab.add(new_token)
            
            merge_record = {
                'iteration': i + 1,
                'pair': f"('{best_pair[0]}', '{best_pair[1]}')",
                'frequency': best_freq,
                'new_token': new_token,
                'vocab_size': len(self.vocab)
            }
            self.merge_history.append(merge_record)
            
            print(f"  반복 {i+1}: {best_pair} → '{new_token}' (빈도: {best_freq})")
        
        print(f"\n[최종 어휘 크기] {len(self.vocab)}")
        return self

In [10]:
tokenizer = BPETokenizer()
tokenizer.train(corpus=corpus, num_merges=10)


[초기 어휘 크기] 23
[병합 시작] 10번 반복

  반복 1: ('h', 'e') → 'he' (빈도: 8)
  반복 2: ('he', 'l') → 'hel' (빈도: 5)
  반복 3: ('hel', 'l') → 'hell' (빈도: 5)
  반복 4: ('hell', 'o') → 'hello' (빈도: 5)
  반복 5: ('hello', '</w>') → 'hello</w>' (빈도: 5)
  반복 6: ('w', 'o') → 'wo' (빈도: 3)
  반복 7: ('wo', 'r') → 'wor' (빈도: 3)
  반복 8: ('wor', 'l') → 'worl' (빈도: 3)
  반복 9: ('worl', 'd') → 'world' (빈도: 3)
  반복 10: ('world', '</w>') → 'world</w>' (빈도: 3)

[최종 어휘 크기] 33


In [11]:
def tokenize_word(word: str, merges: dict, vocab: set) -> List[str]:
    """학습된 merge 규칙을 적용하여 단어 토큰화"""
    # Step 1: 문자 단위로 분해
    chars = list(word) + ['</w>']

    # Step 2: merge 규칙을 순서대로 적용
    for pair, new_token in merges.items():
        while True:
            found = False
            for j in range(len(chars) - 1):
                if chars[j] == pair[0] and chars[j + 1] == pair[1]:
                    chars[j:j + 2] = [new_token]
                    found = True
                    break
            if not found:
                break

    return chars

print("=" * 70)
print("[토큰화 시연]")
print("=" * 70)

#  실제 단어로 테스트
test_words = ["hello", "world", "beautiful", "the", "peace"]

for word in test_words:
    tokens = tokenize_word(word, tokenizer.merges, tokenizer.vocab)
    print(f"\n'{word}' →")

    # 단계별 분해 과정
    chars_initial = list(word) + ['</w>']
    print(f"  1. 초기: {chars_initial}")
    print(f"  2. 최종: {tokens}")
    print(f"  3. 토큰 개수: {len(chars_initial)} → {len(tokens)} (압축: {len(chars_initial) - len(tokens)}개 감소)")

[토큰화 시연]

'hello' →
  1. 초기: ['h', 'e', 'l', 'l', 'o', '</w>']
  2. 최종: ['hello</w>']
  3. 토큰 개수: 6 → 1 (압축: 5개 감소)

'world' →
  1. 초기: ['w', 'o', 'r', 'l', 'd', '</w>']
  2. 최종: ['world</w>']
  3. 토큰 개수: 6 → 1 (압축: 5개 감소)

'beautiful' →
  1. 초기: ['b', 'e', 'a', 'u', 't', 'i', 'f', 'u', 'l', '</w>']
  2. 최종: ['b', 'e', 'a', 'u', 't', 'i', 'f', 'u', 'l', '</w>']
  3. 토큰 개수: 10 → 10 (압축: 0개 감소)

'the' →
  1. 초기: ['t', 'h', 'e', '</w>']
  2. 최종: ['t', 'he', '</w>']
  3. 토큰 개수: 4 → 3 (압축: 1개 감소)

'peace' →
  1. 초기: ['p', 'e', 'a', 'c', 'e', '</w>']
  2. 최종: ['p', 'e', 'a', 'c', 'e', '</w>']
  3. 토큰 개수: 6 → 6 (압축: 0개 감소)
